# Walk-forward NLL — production flow & price models

Self-contained notebook to **load interval data**, run the **current production models**, and evaluate **forward (prequential) NLL**.

| Model | Production class | **Hyperparameters** (edit in §2) |
|-------|------------------|----------------------------------|
| Hurdle lognormal flow | `TorchHurdleLogNormalModel` | `FLOW_HYPERPARAMS` |
| Conditional mechanical mixture price | `TorchConditionalMechanicalMixturePriceModel` | `PRICE_HYPERPARAMS` |

### Hyperparameters vs learned state

| Term | Meaning | Examples | Where to change experiments |
|------|---------|----------|----------------------------|
| **Hyperparameters** | Fixed config from JSON preset; set **before** the walk-forward run | `decay`, `init_p0`, `reference_vwap_vol`, `initial_sigma` | §2 `FLOW_HYPERPARAMS` / `PRICE_HYPERPARAMS` |
| **Learned state** | Online EWMA/RLS quantities updated **each interval** during the run | `p0_buy`, `mu_buy`, `gate_beta`, `market_beta` | Not set manually — output of `get_params()` after evaluation |

This notebook uses **production hyperparameters** from `mm/configs/market_maker.runtime.example.json`. It does **not** load learned state from Postgres; models start cold and learn from the loaded history.

**Evaluation contract** (same as live runtime):
1. At step `t`, score using **learned state** after steps `0..t-1` (hyperparameters fixed).
2. Then `update(previous, current)`.
3. Lower **mean NLL** = better model (compare within flow vs price family).

**To experiment:** change hyperparameters in §2, or swap model builders in §9.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None  # optional — tables/plots still work without it

# Repo root (market-maker/)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "mm").is_dir():
    # notebooks/ -> model_lab/ -> research/ -> repo
    for parent in [Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
        if (parent / "mm").is_dir():
            REPO_ROOT = parent
            break
if not (REPO_ROOT / "mm").is_dir():
    raise RuntimeError("Run with cwd = repo root or research/model_lab/notebooks/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mm.env_loader import load_mm_env

load_mm_env(REPO_ROOT / ".env")

from research.model_lab.core.walk_forward import run_walk_forward
from research.model_lab.data.loader import load_interval_steps, parse_since
from research.model_lab.models.registry import build_flow_model, build_price_model

print("Repo:", REPO_ROOT)

## 2. Hyperparameters (edit here for experiments)

These are **hyperparameters** — the same keys as `controller.local_market_flow_model` and `controller.local_price_model` in the runtime JSON. They are **not** the online learned state (`p0_buy`, `gate_beta`, …).

Defaults match `mm/configs/market_maker.runtime.example.json` → `conservative_local` preset.

In [ ]:
DATA_CONFIG = {
    "exchange": "nobitex",
    "symbol": "USDT_IRT",
    "interval_ms": 30_000,
    "since": "48h",  # or "7d", ISO timestamp
    "source": "auto",  # auto | events | market_facts (price needs full book)
}

# Hyperparameters: production local_market_flow_model
FLOW_HYPERPARAMS = {
    "kind": "hurdle_lognormal",
    "decay": 0.95,
    "init_p0": 0.3,
    "init_mu": 0.0,
    "init_sigma": 1.0,
}

# Hyperparameters: production local_price_model
PRICE_HYPERPARAMS = {
    "kind": "conditional_mechanical_mixture",
    "reference_vwap_vol": 1000.0,
    "shape_alpha": 0.5,
    "decay": 0.99,
    "initial_sigma": 50.0,
    "initial_log_sigma": 0.25,
    "student_df": 3.0,
    "point_mode": "sample",
}

MIN_UPDATES_BEFORE_SCORE = 1

# Back-compat aliases (optional)
FLOW_CONFIG = FLOW_HYPERPARAMS
PRICE_CONFIG = PRICE_HYPERPARAMS

## 3. Load interval steps from Postgres

In [ ]:
until = datetime.now(timezone.utc)
since = parse_since(DATA_CONFIG["since"], until=until)

steps = load_interval_steps(
    exchange=DATA_CONFIG["exchange"],
    symbol=DATA_CONFIG["symbol"],
    interval_ms=DATA_CONFIG["interval_ms"],
    since=since,
    until=until,
    source=DATA_CONFIG["source"],
)

print(f"Loaded {len(steps)} steps")
print(f"  {steps[0].ts} -> {steps[-1].ts}")
print(f"  mid range: {min(s.mid for s in steps):.0f} .. {max(s.mid for s in steps):.0f}")
zero_vol = sum(1 for s in steps if s.buy_volume + s.sell_volume == 0)
print(f"  zero-volume buckets: {zero_vol} ({100*zero_vol/len(steps):.1f}%)")

assert len(steps) >= 2, "Need at least 2 intervals — check MM_DATABASE_URL and persistence"

## 4. Model reference (production implementations)

### Hurdle lognormal flow (`TorchHurdleLogNormalModel`)
- **File:** `mm/mpc/market_order_flow_torch.py`
- Per side: `P(V=0)=p0`, else `V ~ round(LogNormal(μ, σ))`
- Online EWMA on `p0`, and on log-volume moments when `V>0`
- **NLL:** `score_market_flow()` in `mm/models/scoring.py`

### Conditional mechanical mixture price (`TorchConditionalMechanicalMixturePriceModel`)
- **File:** `mm/mpc/price_torch.py`
- Splits each transition into market-driven vs refresh branches
- Gate: `q_market = sigmoid(β0 + β1 * log(1 + buy_vol + sell_vol))`
- Predicts deltas in `(mid, log_spread, log_buy_shape, log_sell_shape)`
- **NLL:** mixture log-likelihood in `research/model_lab/core/scoring.py`

## 5. Walk-forward evaluation helpers

In [ ]:
def scores_to_rows(result) -> list[dict]:
    return [
        {
            "step_index": s.step_index,
            "ts": s.ts,
            "nll": s.nll,
            "log_likelihood": s.log_likelihood,
            "skip_reason": s.skip_reason,
            "std_residual": s.standardized_residual,
        }
        for s in result.scores
    ]


def scores_to_frame(result):
    rows = scores_to_rows(result)
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


def print_summary(name: str, result) -> None:
    s = result.summary
    print(f"\n=== {name} ===")
    print(f"  scored: {s.n_scored} / {s.n_steps}  skipped: {s.n_skipped}")
    print(f"  mean_nll:   {s.mean_nll}")
    print(f"  median_nll: {s.median_nll}")
    if s.skip_reasons:
        print(f"  skip_reasons: {s.skip_reasons}")

## 6. Evaluate hurdle lognormal flow model

In [ ]:
flow_model = build_flow_model(
    kind=FLOW_HYPERPARAMS["kind"],
    **{k: v for k, v in FLOW_HYPERPARAMS.items() if k != "kind"},
)

flow_result = run_walk_forward(
    flow_model,
    steps,
    min_updates_before_score=MIN_UPDATES_BEFORE_SCORE,
)

print_summary("Hurdle lognormal flow", flow_result)
flow_view = scores_to_frame(flow_result)
if pd is not None:
    display(flow_view.head())
else:
    print("First 5 scores:", flow_view[:5])

In [ ]:
# Learned state after walk-forward (not hyperparameters — for inspection only)
flow_learned_state = flow_model.get_params()
print("Flow learned state after full sequence:")
print(json.dumps(flow_learned_state, indent=2))

## 7. Evaluate conditional mechanical mixture price model

In [ ]:
price_model = build_price_model(
    kind=PRICE_HYPERPARAMS["kind"],
    **{k: v for k, v in PRICE_HYPERPARAMS.items() if k != "kind"},
)

price_result = run_walk_forward(
    price_model,
    steps,
    min_updates_before_score=MIN_UPDATES_BEFORE_SCORE,
)

print_summary("Conditional mechanical mixture price", price_result)
price_view = scores_to_frame(price_result)
if pd is not None:
    display(price_view.head())
else:
    print("First 5 scores:", price_view[:5])

In [ ]:
price_learned_state = price_model.get_params()
print("Price learned state after full sequence (truncated):")
for key in ("kind", "gate_beta", "market_prob", "market_beta", "refresh_mean", "market_sigma"):
    if key in price_learned_state:
        print(f"  {key}: {price_learned_state[key]}")

## 8. Summary & optional plots

In [ ]:
leaderboard = [
    {
        "model": flow_result.summary.model_name,
        "family": "flow",
        "mean_nll": flow_result.summary.mean_nll,
        "median_nll": flow_result.summary.median_nll,
        "n_scored": flow_result.summary.n_scored,
    },
    {
        "model": price_result.summary.model_name,
        "family": "price",
        "mean_nll": price_result.summary.mean_nll,
        "median_nll": price_result.summary.median_nll,
        "n_scored": price_result.summary.n_scored,
    },
]
if pd is not None:
    display(pd.DataFrame(leaderboard))
else:
    print(json.dumps(leaderboard, indent=2))

In [ ]:
try:
    import matplotlib.pyplot as plt

    def _plot_nll(rows, title: str, ax) -> None:
        xs, ys = [], []
        for r in rows:
            if r.get("nll") is not None:
                xs.append(r["ts"])
                ys.append(r["nll"])
        ax.plot(xs, ys, linewidth=0.8, alpha=0.8)
        ax.set_ylabel("NLL")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)

    flow_rows = scores_to_rows(flow_result)
    price_rows = scores_to_rows(price_result)
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    _plot_nll(flow_rows, "Flow NLL per step", axes[0])
    _plot_nll(price_rows, "Price NLL per step", axes[1])
    axes[1].set_xlabel("Time")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skip plots (pip install matplotlib)")

## 9. Swap models for experiments

Replace the builders below. Examples:

```python
# Different flow decay
flow_model = build_flow_model(decay=0.99, init_p0=0.2)

# Residual price instead of mixture
from research.model_lab.models.registry import build_price_model
price_model = build_price_model(kind="residual", residual_decay=0.95, residual_sigma=50.0)

# Or use generic spec strings
from research.model_lab.models.registry import build_model
flow_model = build_model("flow:hurdle_gamma")
price_model = build_model("price:residual")
```

Then re-run `run_walk_forward(model, steps)` and compare `summary.mean_nll`.

In [ ]:
# EXPERIMENT SLOT — copy/paste alternative models here
# example:
# alt_flow = build_flow_model(decay=0.99)
# alt_result = run_walk_forward(alt_flow, steps)
# print_summary("alt flow", alt_result)